# Network Designer Example

This notebook demonstrates how to use the Network Designer programmatically.

For the interactive dashboard, use: `python start_network_designer.py`

In [ ]:
# Imports
from pathlib import Path
from energis.io.network_designer import create_network_designer
import yaml

## 1. Create Network Programmatically

In [ ]:
# Create designer instance
designer = create_network_designer()

print(f"Network Designer created")
print(f"Components: {len(designer.components)}")
print(f"Connections: {len(designer.connections)}")

## 2. Add Components

In [ ]:
# Add heat pumps
designer.add_component(x=100, y=300, comp_type='heat_pump')
designer.add_component(x=100, y=500, comp_type='heat_pump')

# Add storage
designer.add_component(x=400, y=400, comp_type='storage')

# Add boiler (existing)
designer.add_component(x=100, y=100, comp_type='boiler')

# Add consumer
designer.add_component(x=700, y=400, comp_type='consumer')

print(f"Added {len(designer.components)} components:")
for comp in designer.components:
    print(f"  - {comp.component_id}: {comp.component_type} at ({comp.x}, {comp.y})")

## 3. Configure Components

In [ ]:
# Configure first heat pump as investment
hp1 = designer.components[0]
hp1.status = 'investment'
hp1.properties['capacity_mw'] = 15.0
hp1.properties['cop'] = 3.8

# Configure second heat pump as existing
hp2 = designer.components[1]
hp2.status = 'existing'
hp2.properties['capacity_mw'] = 10.0
hp2.properties['cop'] = 3.5

# Configure storage as investment
storage = designer.components[2]
storage.status = 'investment'
storage.properties['capacity_mwh'] = 75.0
storage.properties['efficiency'] = 0.98

# Configure boiler as existing
boiler = designer.components[3]
boiler.status = 'existing'
boiler.properties['capacity_mw'] = 20.0
boiler.properties['efficiency'] = 0.95

print("Components configured:")
for comp in designer.components:
    print(f"  {comp.component_id}: {comp.status} - {comp.properties}")

## 4. Add Connections

In [ ]:
# Connect components
hp1_id = designer.components[0].component_id
hp2_id = designer.components[1].component_id
storage_id = designer.components[2].component_id
boiler_id = designer.components[3].component_id
consumer_id = designer.components[4].component_id

# Heat pumps -> Storage
designer.add_connection(hp1_id, storage_id)
designer.add_connection(hp2_id, storage_id)

# Boiler -> Storage
designer.add_connection(boiler_id, storage_id)

# Storage -> Consumer
designer.add_connection(storage_id, consumer_id)

print(f"Added {len(designer.connections)} connections:")
for conn in designer.connections:
    print(f"  {conn.from_id} -> {conn.to_id}")

## 5. Validate Network

In [ ]:
valid, errors = designer.validate_network()

if valid:
    print("✅ Network validation successful!")
else:
    print("❌ Network validation failed:")
    for err in errors:
        print(f"  {err}")

## 6. Export to YAML

In [ ]:
# Export network
output_path = Path('exports/example_network.yaml')
output_path.parent.mkdir(parents=True, exist_ok=True)

config = designer.export_to_yaml(output_path)

print(f"✅ Exported to: {output_path}")
print(f"\nConfig structure:")
print(f"  - {len(config.get('system', {}).get('heat_pumps', []))} heat pumps")
print(f"  - {1 if config.get('system', {}).get('storage', {}).get('enabled') else 0} storage")
print(f"  - {len(config.get('system', {}).get('generators', {}))} generators")

## 7. Display YAML (Preview)

In [ ]:
# Display generated YAML
print("Generated YAML configuration:\n")
print("=" * 60)
print(yaml.dump(config, default_flow_style=False, sort_keys=False)[:2000])  # First 2000 chars
print("...")
print("=" * 60)

## 8. Import from YAML

In [ ]:
# Create new designer and import
designer2 = create_network_designer()
designer2.import_from_yaml(output_path)

print(f"✅ Imported network from {output_path}")
print(f"Components: {len(designer2.components)}")
print(f"Connections: {len(designer2.connections)}")

# Verify
for comp in designer2.components:
    print(f"  - {comp.component_id}: {comp.component_type} ({comp.status})")

## 9. Run Simulation (Optional)

**Note:** This requires:
- Gurobi solver installed and licensed
- Timeseries data (waermebedarf_MWth, strompreis_EUR_MWh, etc.)
- Base configuration files

For a complete simulation, merge with base configs:
```python
from energis.run.rolling_horizon import run_workflow

result = run_workflow([
    'configs/base.yaml',
    'exports/example_network.yaml'
])
```

## 10. Interactive Dashboard

To use the full interactive dashboard with visualization:

```bash
python start_network_designer.py
```

Then open browser at: http://localhost:5006